In [45]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.metrics import log_loss,classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
import os
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/IrrigationNeed/')

In [3]:
irrigation = pd.read_csv("train.csv", index_col = 0)
irrigation

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High


In [4]:
irrigation.isna().sum().sum()

0

In [7]:
le = LabelEncoder()

In [9]:
irrigation["Irrigation_Need"] = le.fit_transform(irrigation["Irrigation_Need"])

In [10]:
X, y = irrigation.drop("Irrigation_Need", axis = 1), irrigation["Irrigation_Need"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26,stratify=irrigation['Irrigation_Need'])

In [11]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                               ).set_output(transform='pandas')

In [12]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [13]:
ss = StandardScaler()

In [14]:
X_train_scaled = ss.fit_transform(X_train_ohe)
X_test_scaled = ss.transform(X_test_ohe)

In [15]:
c = np.linspace(0.01,10,20)
scores = []
for i in tqdm(c):
    lr = LogisticRegression(solver='lbfgs',C=i)
    lr.fit(X_train_scaled,y_train)
    y_pred_prob = lr.predict_proba(X_test_scaled)
    scores.append(['lbfgs',i,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Solver','C','Log Loss'])
df_scores.sort_values('Log Loss')

100%|███████████████████████████████████████████| 20/20 [06:05<00:00, 18.30s/it]


,Solver,C,Log Loss
6,lbfgs,3.164737,0.286624
5,lbfgs,2.638947,0.286624
7,lbfgs,3.690526,0.286624
8,lbfgs,4.216316,0.286624
9,lbfgs,4.742105,0.286624
10,lbfgs,5.267895,0.286624
4,lbfgs,2.113158,0.286624
11,lbfgs,5.793684,0.286624
12,lbfgs,6.319474,0.286624
13,lbfgs,6.845263,0.286624


In [16]:
c = np.linspace(0.01,10,20)
scores = []
for i in tqdm(c):
    lr = LogisticRegression(solver='newton-cg',C=i)
    lr.fit(X_train_scaled,y_train)
    y_pred_prob = lr.predict_proba(X_test_scaled)
    scores.append(['newton-cg',i,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Solver','C','Log Loss'])
df_scores.sort_values('Log Loss')

 10%|████▍                                       | 2/20 [02:03<18:37, 62.07s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/scipy/optimize/_linesearch.py:306: LineSearchWarning: The line search algorithm did not converge
  warn('The line search algorithm did not converge', LineSearchWarning)
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/utils/optimize.py:204: UserWarning: Line Search failed
  warnings.warn("Line Search failed")
 15%|██████▌                                     | 3/20 [03:24<20:02, 70.75s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/scipy/optimize/_linesearch.py:306: LineSearchWarning: The line search algorithm did not converge
  warn('The line search algorithm did not converge', LineSearchWarning)
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/utils/optimize.py:204: UserWarning: Line Search failed
  warnings.warn("Line Search failed")
100%|███████████████████████████████████████████| 20/20 [15:05<00:00, 45.26s/it]


,Solver,C,Log Loss
7,newton-cg,3.690526,0.286623
8,newton-cg,4.216316,0.286623
6,newton-cg,3.164737,0.286623
9,newton-cg,4.742105,0.286623
10,newton-cg,5.267895,0.286623
11,newton-cg,5.793684,0.286623
12,newton-cg,6.319474,0.286623
5,newton-cg,2.638947,0.286623
13,newton-cg,6.845263,0.286623
14,newton-cg,7.371053,0.286623


In [17]:
c = np.linspace(0.01,10,20)
scores = []
for i in tqdm(c):
    lr = LogisticRegression(solver='newton-cholesky',C=i)
    lr.fit(X_train_scaled,y_train)
    y_pred_prob = lr.predict_proba(X_test_scaled)
    scores.append(['newton-cholesky',i,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Solver','C','Log Loss'])
df_scores.sort_values('Log Loss')

100%|███████████████████████████████████████████| 20/20 [01:17<00:00,  3.86s/it]


,Solver,C,Log Loss
19,newton-cholesky,10.000000,0.308963
18,newton-cholesky,9.474211,0.308963
17,newton-cholesky,8.948421,0.308963
16,newton-cholesky,8.422632,0.308964
15,newton-cholesky,7.896842,0.308964
14,newton-cholesky,7.371053,0.308964
13,newton-cholesky,6.845263,0.308964
12,newton-cholesky,6.319474,0.308964
11,newton-cholesky,5.793684,0.308964
10,newton-cholesky,5.267895,0.308964


In [18]:
c = np.linspace(0.01,10,20)
scores = []
for i in tqdm(c):
    lr = LogisticRegression(solver='sag',C=i)
    lr.fit(X_train_scaled,y_train)
    y_pred_prob = lr.predict_proba(X_test_scaled)
    scores.append(['sag',i,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Solver','C','Log Loss'])
df_scores.sort_values('Log Loss')

 10%|████▍                                       | 2/20 [00:44<07:34, 25.25s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
 15%|██████▌                                     | 3/20 [01:26<09:15, 32.70s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
 20%|████████▊                                   | 4/20 [02:08<09:39, 36.23s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
 25%|███████████                                 | 5/20 [02:49<09:32, 38.15s/it]/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached 

,Solver,C,Log Loss
11,newton-cholesky,5.793684,0.286619
17,newton-cholesky,8.948421,0.286620
14,newton-cholesky,7.371053,0.286622
8,newton-cholesky,4.216316,0.286622
3,newton-cholesky,1.587368,0.286622
18,newton-cholesky,9.474211,0.286623
13,newton-cholesky,6.845263,0.286623
4,newton-cholesky,2.113158,0.286623
2,newton-cholesky,1.061579,0.286624
6,newton-cholesky,3.164737,0.286624


In [19]:
c = np.linspace(0.01,10,20)
scores = []
for i in tqdm(c):
    lr = LogisticRegression(solver='saga',C=i)
    lr.fit(X_train_scaled,y_train)
    y_pred_prob = lr.predict_proba(X_test_scaled)
    scores.append(['saga',i,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['Solver','C','Log Loss'])
df_scores.sort_values('Log Loss')

100%|███████████████████████████████████████████| 20/20 [03:26<00:00, 10.34s/it]


,Solver,C,Log Loss
17,saga,8.948421,0.286623
11,saga,5.793684,0.286623
13,saga,6.845263,0.286623
18,saga,9.474211,0.286623
4,saga,2.113158,0.286623
9,saga,4.742105,0.286623
5,saga,2.638947,0.286623
7,saga,3.690526,0.286623
1,saga,0.535789,0.286623
19,saga,10.000000,0.286623


In [42]:
bm_train = LogisticRegression(C=5.793684,solver='newton-cholesky')
bm.fit(X_train_scaled,y_train)

LogisticRegression(C=5.793684, solver='newton-cholesky')

In [43]:
y_pred = bm.predict(X_test_scaled)

In [46]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.04      0.07      6303
           1       0.90      0.92      0.91    110975
           2       0.80      0.84      0.82     71722

    accuracy                           0.86    189000
   macro avg       0.90      0.60      0.60    189000
weighted avg       0.86      0.86      0.85    189000



In [21]:
X_ohe = transformer.fit_transform(X)
X_scaled = ss.fit_transform(X_ohe)

In [23]:
bm = LogisticRegression(C=5.793684,solver='newton-cholesky')
bm.fit(X_scaled,y)

LogisticRegression(C=5.793684, solver='newton-cholesky')

In [25]:
test = pd.read_csv('test.csv',index_col=0)
test

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,Yes,43.49,East
899996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,Yes,92.03,West
899997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,Yes,34.02,East


In [27]:
test_ohe = transformer.transform(test)
test_scaled = ss.transform(test_ohe)

In [31]:
y_pred = bm.predict(test_scaled)
y_pred_labels = le.inverse_transform(y_pred)
y_pred_labels

array(['Low', 'Low', 'Low', ..., 'Medium', 'Low', 'Medium'], dtype=object)

In [32]:
sample = pd.read_csv('sample_submission.csv')

In [34]:
sample['Irrigation_Need'] = y_pred_labels

In [35]:
sample

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Low
269996,899996,Low
269997,899997,Medium
269998,899998,Low


In [36]:
sample.to_csv("SubmissionKaggle.csv",index=False)

Irrigation_Need
Low       162866
Medium    106856
High         278
Name: count, dtype: int64